# KAN: «формула выживания» на Титанике

Обычная нейросеть (MLP) учит **числа**-веса, а нелинейность у неё фиксированная
(ReLU и т.п.). KAN (Kolmogorov–Arnold Network) устроена наоборот: обучаемая
часть — это **маленькие функции одной переменной** (сплайны) на рёбрах сети,
а нейрон просто складывает их результаты.

Чем это интересно:

- каждую выученную функцию можно нарисовать и посмотреть глазами;
- рёбра с нулевым вкладом можно отрезать (pruning);
- в конце сеть можно попробовать превратить в обычную формулу (symbolic regression).

План:

1. Спроектировать представление данных — KAN хочет осмысленные числа на входе.
2. Обучить максимально маленькую сеть: один KAN-слой, 4 признака.
3. Убедиться по метрикам, что она не переобучена и не недообучена.
4. Посмотреть, какие функции сеть выучила.
5. Добавить FamilySize, затем IsAlone — стало ли лучше?
6. Pruning + symbolic regression → формула выживания.

## 1. Представление данных

Каждая входная функция KAN — кривая над числовой осью, поэтому вход должен быть
числом, у которого «больше/меньше» имеет смысл. Категория, закодированная
произвольными числами (например, порт посадки как 0/1/2), даст сети
бессмысленную ось. Наши признаки:

| Признак | Как кодируем | Почему |
|---|---|---|
| Age | как есть; пропуски — медианой по train | возраст — настоящее число |
| Fare → LogFare | log1p(Fare) | цена скошена: большинство < 50, хвост до 512 — логарифм сжимает хвост |
| Pclass | 1/2/3 как есть | порядковая шкала: 1-й класс лучше 3-го |
| Sex | 0 = мужчина, 1 = женщина | бинарный признак — уже число |
| FamilySize | SibSp + Parch + 1 | для эксперимента 2 |
| IsAlone | 1, если FamilySize == 1 | для эксперимента 3 |

Весь код подготовки — в [titanic_data.py](titanic_data.py), обучение, метрики
и графики — в [kan_helpers.py](kan_helpers.py).

In [1]:
import warnings

import pandas as pd
from kan import KAN
from kan.utils import ex_round

from titanic_data import load_features, make_dataset
from kan_helpers import (train, metrics_table, plot_training,
                         plot_edge_functions, plot_feature_scores,
                         prune_features)

warnings.filterwarnings("ignore", message="Converting a tensor")  # шумное, но безобидное

X, y = load_features("Titanic-Dataset.csv")
X.head()

,Age,LogFare,Pclass,Sex,FamilySize,IsAlone
0,22.0,2.110213,3,0,2,0
1,38.0,4.280593,1,1,2,0
2,26.0,2.188856,3,1,1,1
3,35.0,3.990834,1,1,2,0
4,35.0,2.202765,3,0,1,1


## 2. Эксперимент 1: один KAN-слой, 4 признака

Сеть `width=[4, 1]`: 4 входа, 1 выход, между ними один KAN-слой. Выход — просто
сумма четырёх выученных функций:

**score = φ₁(Age) + φ₂(LogFare) + φ₃(Pclass) + φ₄(Sex)**,  P(выжил) = σ(score)

Такая модель называется **аддитивной**: признаки не взаимодействуют, каждый
вносит независимый вклад — зато каждый вклад можно нарисовать.

Параметры:

- `grid=3` — у сплайнов мало узлов → функции гладкие, меньше шансов переобучиться;
- `k=3` — кубические сплайны (стандарт);
- `lamb=0.001` — лёгкая регуляризация: прижимает к нулю рёбра, которые не помогают;
- `auto_save=False` — чтобы pykan не писал чекпоинты на диск.

In [2]:
FEATURES = ["Age", "LogFare", "Pclass", "Sex"]
dataset = make_dataset(X, y, FEATURES)

model = KAN(width=[4, 1], grid=3, k=3, seed=42, auto_save=False)
results = train(model, dataset, steps=30, lamb=0.001)
plot_training(results)

Обе кривые быстро падают и выходят на полку — LBFGS сходится за несколько шагов.
Ближе к концу test-кривая может слегка подрастать, пока train ещё чуть падает —
лёгкий намёк на переобучение, но разрыв остаётся небольшим и стабильным.

In [3]:
metrics_table(model, dataset)

,Accuracy,F1,Recall,ROC-AUC,LogLoss
train,0.819,0.758,0.740,0.864,0.425
test,0.799,0.735,0.725,0.833,0.505


Как читать метрики:

- **Accuracy ≈ 0.80** — доля верных ответов. Для сравнения: модель «все погибли»
  даёт 0.62 — мы заметно лучше.
- **F1 ≈ 0.74** и **Recall ≈ 0.72** — насколько хорошо находим именно выживших
  (меньший класс, поэтому Accuracy одной недостаточно).
- **ROC-AUC ≈ 0.83** — качество ранжирования по вероятности; не зависит от порога 0.5.
- **LogLoss** — штраф за плохие вероятности: train ≈ 0.43 против test ≈ 0.51.
  Разрыв умеренный: лёгкое переобучение есть, катастрофы нет. Константная модель
  (всем P = 0.38 — доля выживших) имела бы ≈ 0.66.

Вывод: сеть из одного KAN-слоя не переобучена, не недообучена и уже играет на
уровне классических моделей из week2.

*Замечание: LBFGS на CPU не бит-детерминирован — при перезапуске ноутбука
метрики могут гулять во 2–3 знаке. Это нормально.*

### А нужен ли второй слой?

Аддитивная модель не умеет взаимодействий (например, «женщина **и** 3-й класс»).
Проверим сеть `[4, 2, 1]` — с двумя нейронами в скрытом слое.

In [4]:
model2 = KAN(width=[4, 2, 1], grid=3, k=3, seed=42, auto_save=False)
train(model2, dataset, steps=30, lamb=0.001)
metrics_table(model2, dataset)

,Accuracy,F1,Recall,ROC-AUC,LogLoss
train,0.838,0.774,0.722,0.891,0.384
test,0.804,0.720,0.652,0.845,0.461


Реального выигрыша нет: на test какие-то метрики чуть выше, какие-то ниже, зато
разрыв train/test заметно вырос (сравните LogLoss и ROC-AUC между строками) —
сеть стала мощнее и сильнее подстраивается под обучающую выборку.

Оставляем один слой: чем меньше сеть, тем легче её читать.

## 3. Что выучила сеть

У KAN обучаются не числа, а функции — достанем каждую φ и нарисуем. По оси X —
значение признака, по оси Y — его вклад в score: чем выше кривая, тем больше
шансов выжить.

In [5]:
plot_edge_functions(model, dataset, FEATURES)

Абсолютные значения по оси Y не важны (общий сдвиг уходит в константу) — смысл
несут форма кривой и перепады высоты. Что видно:

- **Sex** — всего два значения, но перепад самый большой: ≈ +2.7 к score
  у женщин относительно мужчин. Главный фактор.
- **Pclass** — почти прямая вниз: каждый следующий класс отнимает ≈ 1.4.
- **Age** — единственная по-настоящему нелинейная кривая: заметный бонус
  самым маленьким (левый край на ≈ 3 выше «взрослого» уровня), после ~16 лет —
  почти плато с мелкими волнами. Это тот самый эффект «дети выживают», который
  линейная модель без флага IsChild не видит (мы это обсуждали в `todo.txt`).
  Правому краю (около 80 лет) доверять не стоит — там сплайн опирается на
  единичные точки.
- **LogFare** — резкий скачок между «билет почти даром» (LogFare ≈ 0) и
  обычными билетами, дальше почти плато. Факт из данных: пассажиров с Fare = 0
  в выборке 15, все мужчины, выжил один.

In [6]:
plot_feature_scores(model, FEATURES)

Важности подтверждают картинку: Sex и Pclass — главные, Age и LogFare —
вспомогательные. Это согласуется с EDA из week2. Попробуем дать сети больше
информации.

## 4. Эксперименты 2 и 3: добавляем FamilySize и IsAlone

Правило из `todo.txt`: один эксперимент — одно изменение. Обучаем ту же
архитектуру `[N, 1]` на расширенных наборах и сравниваем метрики на test.

In [7]:
experiments = {"база (4 признака)": (model, dataset, FEATURES)}

for extra in (["FamilySize"], ["FamilySize", "IsAlone"]):
    feats = FEATURES + extra
    ds = make_dataset(X, y, feats)
    m = KAN(width=[len(feats), 1], grid=3, k=3, seed=42, auto_save=False)
    train(m, ds, steps=30, lamb=0.001)
    experiments["+" + " +".join(extra)] = (m, ds, feats)

pd.DataFrame({name: metrics_table(m, ds).loc["test"]
              for name, (m, ds, feats) in experiments.items()}).T

,Accuracy,F1,Recall,ROC-AUC,LogLoss
база (4 признака),0.799,0.735,0.725,0.833,0.505
+FamilySize,0.827,0.763,0.725,0.851,0.469
+FamilySize +IsAlone,0.816,0.744,0.696,0.846,0.467


FamilySize улучшает все метрики разом — зависимость выживаемости от размера
семьи мы уже видели в week2, и сети она пригодилась. IsAlone поверх FamilySize
не добавляет ничего (метрики те же или чуть хуже), и это ожидаемо: IsAlone
вычисляется из FamilySize, новой информации в нём нет.

## 5. Pruning: отрезаем лишнее

Спросим у сети с шестью входами, какие из них она реально использует.

In [8]:
model6, dataset6, features6 = experiments["+FamilySize +IsAlone"]
plot_feature_scores(model6, features6)

Важность IsAlone — ноль: регуляризация прижала его рёбра к нулю, сеть этот вход
игнорирует. `prune_input` физически убирает такие входы из сети.

In [9]:
final_model, final_features = prune_features(model6, features6, threshold=0.03)
print("Остались:", final_features)
metrics_table(final_model, dataset6)

keep: [True, True, True, True, True, False]
Остались: ['Age', 'LogFare', 'Pclass', 'Sex', 'FamilySize']


,Accuracy,F1,Recall,ROC-AUC,LogLoss
train,0.816,0.748,0.711,0.871,0.419
test,0.821,0.746,0.681,0.846,0.470


Метрики сдвинулись разве что в третьем знаке — мы отрезали вход, чей вклад и
так был практически нулевым. Дообучать сеть после такого pruning не нужно.

## 6. Symbolic regression: из сети — в формулу

pykan берёт каждую выученную φ и подбирает к ней простую формулу вида
`c·f(a·x + b) + d`, перебирая f из библиотеки (у нас: x, x², tanh, exp) и
оценивая подгонку по R² — эти строки печатаются ниже. После замены всех φ
короткое дообучение подгоняет коэффициенты.

In [10]:
final_model.auto_symbolic(lib=["x", "x^2", "tanh", "exp"])
train(final_model, dataset6, steps=20)  # дообучаем коэффициенты формулы
formula = ex_round(final_model.symbolic_formula(var=final_features)[0][0], 3)
formula

fixing (0,0,0) with x, r2=0.633472204208374, c=1


fixing (0,1,0) with x, r2=0.6314492225646973, c=1


fixing (0,2,0) with exp, r2=1.0000007152557373, c=2


fixing (0,3,0) with x, r2=1.0000007152557373, c=1
fixing (0,4,0) with x^2, r2=0.9921137094497681, c=2


-0.038*Age + 0.429*LogFare + 2.679*Sex - 0.001*(4.921 - 6.8*FamilySize)**2 - 3.785 + 4.707*exp(-0.335*Pclass)

In [11]:
metrics_table(final_model, dataset6)

,Accuracy,F1,Recall,ROC-AUC,LogLoss
train,0.794,0.722,0.700,0.865,0.432
test,0.827,0.770,0.754,0.848,0.456


**P(выжил) = σ(score)** — сигмоида переводит score в вероятность: score > 0
означает P > 0.5, то есть «скорее выжил». Что говорит формула:

- `+2.679 · Sex` — женщина получает +2.7 к score. Главный фактор: перекрыть
  его минусами почти нечем.
- `+4.707 · e^(−0.335·Pclass)` — вклад класса: 1-й ≈ +3.4, 2-й ≈ +2.4,
  3-й ≈ +1.7. На трёх точках {1, 2, 3} экспонента и прямая почти неразличимы —
  pykan выбрал exp из-за микроскопически лучшего R²; читается это просто как
  «каждый класс ниже — минус 0.8–1.0».
- `−0.038 · Age` — каждый год возраста немного минусует (≈ −0.4 за 10 лет).
- `−0.001 · (4.921 − 6.8·FamilySize)²` — раскрыв скобки: ≈ −0.046·(FamilySize − 0.72)².
  Штраф растёт квадратично с размером семьи: семья из 2 человек не теряет почти
  ничего (−0.08), из 5 — уже −0.8, из 8 — −2.4.
- `+0.429 · LogFare` — дороже билет — выше шанс.

Строки `fixing ... r2=...` выше показывают, где формула честная, а где
компромисс: Sex и Pclass подогнаны идеально (R² ≈ 1), FamilySize — почти
(0.99), а Age и LogFare — лишь на ≈ 0.63: их настоящие кривые (детский бонус,
скачок у бесплатных билетов) к прямой не сводятся. При этом метрики в таблице
показывают, что формула не хуже сплайновой сети — на test даже чуть лучше:
лишняя гибкость ушла, суть осталась. (То, что test здесь лучше train — обычное
дело на маленькой выборке, а не заслуга модели.)

Получилась, по сути, **логистическая регрессия** — линейная комбинация
признаков под сигмоидой. Но есть разница: мы её не постулировали заранее, а
*вывели из сети*. KAN сначала выучила свободные кривые, показала, что почти все
они — прямые, и заодно показала, где именно прямой недостаточно (Age, LogFare).

*(При перезапуске pykan может выбрать другую запись той же зависимости —
например, прямую вместо exp для Pclass: на трёх точках они совпадают.)*

## 7. Подбор порога классификации

До сих пор мы превращали вероятность в ответ порогом 0.5. Но 0.5 — не догма:
двигая порог, мы обмениваем **Precision** (как часто предсказание «выжил»
оказывается правдой) на **Recall** (какую долю настоящих выживших мы нашли).

Важное правило: порог — такой же настраиваемый параметр, как и остальные.
Подбираем его по train, а проверяем на test.

In [12]:
from kan_helpers import plot_threshold_curves, threshold_table

fig, best_t = plot_threshold_curves(final_model, dataset6)
fig

In [13]:
threshold_table(final_model, dataset6, [0.5, best_t])

,Accuracy,Precision,Recall,F1
порог 0.50,0.827,0.788,0.754,0.770
порог 0.36,0.782,0.688,0.797,0.738


Обмен сработал как задумано: при пороге 0.36 Recall на test вырос
(0.754 → 0.797 — находим больше настоящих выживших), а Precision упал
(0.788 → 0.688 — больше ложных тревог).

Но общий F1 на test не улучшился (0.770 → 0.738), хотя на train порог 0.36 был
лучшим. Это второй урок: выигрыш, подобранный по 712 строкам train, не обязан
переноситься на 179 строк test — на таких объёмах порог надо подбирать
кросс-валидацией (об этом в §9).

Порог остаётся полезным инструментом, когда у ошибок разная цена:
«не пропустить выжившего» (поиск и спасение) — двигаем порог вниз ради Recall;
«не объявить погибшего выжившим» — вверх ради Precision.

## 8. Сравнение с классическими моделями

Все модели получают **одни и те же 6 признаков и тот же train/test split**,
что и KAN, — иначе сравнение нечестное. Участники:

- `Dummy (prior)` — базовая линия: всем даёт вероятность 0.38 (долю выживших);
- `LogisticRegression` — прямой конкурент нашей формулы (тоже линейная модель
  под сигмоидой, но постулированная заранее);
- `RandomForest` — представитель ансамблей деревьев, умеет взаимодействия.

In [14]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from kan_helpers import sklearn_metrics_row

classics = {
    "Dummy (prior)": DummyClassifier(strategy="prior"),
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression()),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
}

rows = {name: sklearn_metrics_row(m, dataset6) for name, m in classics.items()}
rows["KAN (сплайны, 5 призн.)"] = metrics_table(*experiments["+FamilySize"][:2]).loc["test"]
rows["KAN (формула)"] = metrics_table(final_model, dataset6).loc["test"]
pd.DataFrame(rows).T

,Accuracy,F1,Recall,ROC-AUC,LogLoss
Dummy (prior),0.615,0.000,0.000,0.500,0.667
LogisticRegression,0.810,0.742,0.710,0.845,0.458
RandomForest,0.816,0.752,0.725,0.845,0.509
"KAN (сплайны, 5 призн.)",0.827,0.763,0.725,0.851,0.469
KAN (формула),0.827,0.770,0.754,0.848,0.456


Обе версии KAN — наверху таблицы: формула даёт лучшие F1, Recall и
LogLoss, сплайновая сеть — лучший ROC-AUC. Два наблюдения:

- **LogisticRegression чуть слабее нашей формулы**, хотя формула — тоже
  линейная модель. Разница в пути: KAN сначала выучила свободные кривые
  (log-ось для Fare, готовые нелинейности) и только потом упростилась,
  «зная», что именно она теряет.
- **RandomForest догоняет по Accuracy, но проигрывает по LogLoss** (0.51):
  лес склонен к переуверенным вероятностям.

И честная оговорка: разница между всеми моделями (кроме Dummy) — единицы
процентов на 179 тестовых строках, то есть в пределах шума одного split.
Уверенно заявлять «KAN лучше» можно только после кросс-валидации — см. §9.

## 9. Как реально превзойти классические модели (план, без реализации)

Урок week2 повторяется: модели идут плотной группой, выигрывать надо данными и
постановкой задачи, а не перебором архитектур. Что даст шанс:

1. **Признаки из очереди `todo.txt`.** Title из Name (Master = мальчик — ловит
   связку Age×Sex одним входом), HasCabin, TicketGroupSize / FarePerPerson.
   KAN сильна именно на осмысленных осях — дать ей новые оси.
2. **Взаимодействия.** Аддитивная сеть `[N, 1]` в принципе не видит «женщина
   **и** 3-й класс». Варианты: подать произведение Sex×Pclass отдельным
   входом; вернуть скрытый слой `[N, 2, 1]`, но уже с новыми признаками и
   подбором `lamb`; у pykan есть и узлы-умножения (`mult_arity`).
3. **Кросс-валидация вместо одного split.** Разница между моделями в таблице —
   1–2 процента, это меньше разброса между фолдами. Подбирать grid / lamb /
   порог надо по CV, иначе мы подгоняемся под конкретные 179 тестовых строк.
4. **Ансамбль из KAN.** LBFGS чувствителен к мелочам (мы это видели по
   плавающему третьему знаку) — усреднение вероятностей нескольких сетей с
   разными seed убирает эту дисперсию почти бесплатно.
5. **Grid refinement.** Приём из pykan: обучить грубую сеть (`grid=3`), затем
   `model.refine()` до grid 5–10 и дообучить — точность сплайнов растёт, а
   стартовать сразу с гибкой (переобучаемой) сети не приходится.

Честная оговорка: потолок на Титанике — примерно 0.83–0.85 Accuracy, дальше
начинается случайность спасения. «Превзойти классику» здесь означает +1–2
процента и более устойчивые метрики, а не прорыв.

## 10. Выводы

- **KAN из одного слоя — это аддитивная модель (GAM):** сумма читаемых функций
  по одному признаку. На Титанике она играет на уровне классических моделей
  (таблица в §8) и при этом полностью прозрачна.
- **Представление данных решает.** Мы дали сети осмысленные оси (порядковый
  Pclass, log от Fare, бинарный Sex) — дальше она справилась сама.
- **FamilySize полезен, IsAlone — мёртвый вход:** важность 0, pruning убирает
  его без малейшей потери качества.
- **Формула выживания** получилась по сути логистической регрессией — но мы её
  не постулировали заранее, а *вывели из сети*, вместе с картой мест, где
  линейность нарушается (Age, LogFare).
- **Порог — управляемый параметр:** сдвиг с 0.5 обменивает Precision на Recall
  под задачу (§7).

Куда двигаться дальше — план в §9.